In [ ]:
import os
import pandas as pd
from chembl_webresource_client.new_client import new_client

ficheiro_csv = 'cdk_family_pretraining_data.csv'

# Verificar se o ficheiro já existe localmente
if os.path.exists(ficheiro_csv):
    print(f"O ficheiro '{ficheiro_csv}' já existe. A carregar os dados localmente...")
    df_family = pd.read_csv(ficheiro_csv)
    
else:
    print("O ficheiro não existe localmente. A descarregar da ChEMBL...")
    
    # IDs de vários alvos da família CDK na ChEMBL para o pré-treino
    cdk_targets = {
        'CDK1': 'CHEMBL302',
        'CDK2': 'CHEMBL301',
        'CDK4': 'CHEMBL359',
        'CDK5': 'CHEMBL303',
        'CDK6': 'CHEMBL2972',
        'CDK9': 'CHEMBL3116'
    }

    activity = new_client.activity
    all_data = []

    for cdk_name, target_id in cdk_targets.items():
        print(f"A descarregar dados para {cdk_name} ({target_id})...")
        res = activity.filter(target_chembl_id=target_id, standard_type__in=['IC50', 'Kd', 'Ki'])
        #res = activity.filter(target_chembl_id=target_id, standard_type__in=['IC50'])
        
        for act in res:
            if act.get('canonical_smiles') and act.get('pchembl_value') is not None:
                all_data.append({
                    'target': cdk_name,
                    'molecule_chembl_id': act['molecule_chembl_id'],
                    'smiles': act['canonical_smiles'],
                    'standard_type': act['standard_type'],
                    'pchembl_value': float(act['pchembl_value'])
                })

    df_family = pd.DataFrame(all_data)
    df_family = df_family.dropna().drop_duplicates(subset=['target', 'molecule_chembl_id'])

    # Converter pChEMBL para pIC50 equivalente
    df_family['pIC50'] = df_family['pchembl_value']

    # Guardar os dados da família para o pré-treino
    df_family.to_csv(ficheiro_csv, index=False)
    print("Download concluído e ficheiro guardado com sucesso!")

# --- A partir daqui corre sempre (quer venha do ficheiro, quer venha do download) ---
print(f"\nTotal de registos na família CDK: {len(df_family)}")
print(df_family['target'].value_counts())
print("\n== Compostos únicos por tipo de medição ==")  #elminar se for só para usar ic50
print(df_family.groupby(['target', 'standard_type'])['molecule_chembl_id']
      .nunique().unstack(fill_value=0))



O ficheiro 'cdk_family_pretraining_data.csv' já existe. A carregar os dados localmente...

Total de registos na família CDK: 6422
target
CDK2    2643
CDK9    1779
CDK1    1777
CDK4     168
CDK5      52
CDK6       3
Name: count, dtype: int64

== Compostos únicos por tipo de medição ==
standard_type  IC50  Kd    Ki
target                       
CDK1             38  14  1725
CDK2           2021  66   556
CDK4            165   3     0
CDK5             52   0     0
CDK6              0   3     0
CDK9           1531  45   203


In [17]:
# Separar especificamente o vosso alvo de fine-tuning (ex: CDK2 ou CDK9)
target_escolhido = 'CDK2' # ou 'CDK9'
df_finetune = df_family[df_family['target'] == target_escolhido]
df_finetune.to_csv(f'{target_escolhido.lower()}_finetune_data.csv', index=False)
print(f"Dados para fine-tuning em {target_escolhido}: {len(df_finetune)} compostos")

Dados para fine-tuning em CDK2: 2643 compostos


In [18]:
# 1. Definir o limiar (threshold) para considerar um composto como "ativo" (ex: pIC50 >= 6.0 ou 7.0)
limiar = 6.0 

# 2. Calcular ativos vs inativos
total_compostos = len(df_finetune)
ativos = (df_finetune['pIC50'] >= limiar).sum()
inativos = total_compostos - ativos

print(f"--- Estatísticas para o alvo selecionado ---")
print(f"Total de compostos únicos: {total_compostos}")
print(f"Ativos (pIC50 >= {limiar}): {ativos} ({ativos/total_compostos:.1%})")
print(f"Inativos (pIC50 < {limiar}): {inativos} ({inativos/total_compostos:.1%})")

# 3. Ver a distribuição rápida por escalões de pIC50
print("\nDistribuição por intervalos de pIC50:")
print(pd.cut(df_finetune['pIC50'], bins=[0, 5, 6, 7, 8, 10], labels=['<5', '5-6', '6-7', '7-8', '>8'],right=False).value_counts().sort_index())

--- Estatísticas para o alvo selecionado ---
Total de compostos únicos: 2643
Ativos (pIC50 >= 6.0): 1774 (67.1%)
Inativos (pIC50 < 6.0): 869 (32.9%)

Distribuição por intervalos de pIC50:
pIC50
<5     214
5-6    655
6-7    822
7-8    664
>8     287
Name: count, dtype: int64



== Compostos únicos por tipo de medição ==
standard_type  IC50
target             
CDK1             45
CDK2           2046
CDK4            165
CDK5             52
CDK9           1544
